## Building Dense Vectors Using PyTorch and Transformers
https://github.com/jamescalam/transformers/blob/main/course/similarity/04_sentence_transformers.ipynb

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch

C:\Users\upadh\anaconda3\envs\NLP\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
C:\Users\upadh\anaconda3\envs\NLP\lib\site-packages\numpy\.libs\libopenblas.FB5AE2TYXYH2IJRDKGDGQ3XBKLKTF43H.gfortran-win_amd64.dll
C:\Users\upadh\anaconda3\envs\NLP\lib\site-packages\numpy\.libs\libopenblas.WCDJNK7YVMPZQ2ME2ZZHJJRJ3JIKNDB7.gfortran-win_amd64.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [7]:
model_name = 'sentence-transformers/bert-base-nli-mean-tokens'

In [8]:
# First we initialize our model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [9]:
text = "hello world what a time to be alive!"

In [10]:
# Now we tokenize(give index to i/p) using the tokenizer
tokens = tokenizer.encode_plus(text, max_length=128, 
                              truncation=True, 
                              padding='max_length',
                              return_tensors='pt')

In [11]:
# as now we tokenized our input, we can pass them through our model to create word embeddings/sentence embeddings
outputs = model(**tokens)

In [14]:
outputs #last_hidden_state

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[ 3.0681e-01, -7.8806e-02,  1.7431e+00,  ..., -2.5348e-02,
          -1.1080e-01,  4.8310e-02],
         [ 7.1302e-01,  1.0437e-01,  1.8346e+00,  ...,  1.1343e-01,
          -7.5563e-02,  1.2668e-01],
         [ 8.1722e-01,  1.1321e-01,  1.5408e+00,  ..., -3.8067e-01,
           8.7478e-02, -1.9020e-01],
         ...,
         [ 5.4669e-01,  1.7181e-01,  1.1392e+00,  ...,  3.8549e-02,
          -1.5396e-01,  2.3015e-01],
         [ 3.4457e-01,  1.3151e-01,  1.1324e+00,  ..., -1.4206e-03,
          -1.7517e-01,  1.5220e-01],
         [ 3.2320e-01,  3.3350e-03,  1.1888e+00,  ...,  1.6736e-02,
          -2.0864e-01,  8.9315e-02]]], grad_fn=<NativeLayerNormBackward>), pooler_output=tensor([[-7.1334e-01, -2.7907e-01,  7.7808e-01, -1.9139e-01, -1.4626e-01,
         -2.2970e-01,  4.3867e-01, -1.3565e-01,  4.1929e-01, -8.8569e-01,
          3.7006e-01, -3.7974e-01,  7.3328e-01, -2.6172e-01,  8.7081e-01,
         -3.7279e-0

In [15]:
# to pull out the embeddings:
embeddings = outputs.last_hidden_state
embeddings

tensor([[[ 3.0681e-01, -7.8806e-02,  1.7431e+00,  ..., -2.5348e-02,
          -1.1080e-01,  4.8310e-02],
         [ 7.1302e-01,  1.0437e-01,  1.8346e+00,  ...,  1.1343e-01,
          -7.5563e-02,  1.2668e-01],
         [ 8.1722e-01,  1.1321e-01,  1.5408e+00,  ..., -3.8067e-01,
           8.7478e-02, -1.9020e-01],
         ...,
         [ 5.4669e-01,  1.7181e-01,  1.1392e+00,  ...,  3.8549e-02,
          -1.5396e-01,  2.3015e-01],
         [ 3.4457e-01,  1.3151e-01,  1.1324e+00,  ..., -1.4206e-03,
          -1.7517e-01,  1.5220e-01],
         [ 3.2320e-01,  3.3350e-03,  1.1888e+00,  ...,  1.6736e-02,
          -2.0864e-01,  8.9315e-02]]], grad_fn=<NativeLayerNormBackward>)

In [17]:
embeddings.shape # 1 tensor of 128 rows and 768 columns
# why 128, number of tokens each of lenght 768 basically each word out of 128 is represented with 768 dimensional vector

torch.Size([1, 128, 768])

In [18]:
# now to create dense vector embedding we will be using mean pooling operation
attention_mask = tokens['attention_mask']
attention_mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]])

In [19]:
# now we need to convert the attention mask to size of the word embeddings which is 128x768
mask = attention_mask.unsqueeze(-1).expand(embeddings.size()).float()
mask.shape

torch.Size([1, 128, 768])

In [20]:
masked_embeddings = embeddings * mask
masked_embeddings.shape

torch.Size([1, 128, 768])

In [23]:
masked_embeddings # starting len(text) will be non-zero and rest will be zero

tensor([[[ 0.3068, -0.0788,  1.7431,  ..., -0.0253, -0.1108,  0.0483],
         [ 0.7130,  0.1044,  1.8346,  ...,  0.1134, -0.0756,  0.1267],
         [ 0.8172,  0.1132,  1.5408,  ..., -0.3807,  0.0875, -0.1902],
         ...,
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000, -0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ..., -0.0000, -0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000, -0.0000,  0.0000]]],
       grad_fn=<MulBackward0>)

In [33]:
# Then we sum the remained of the embeddings along axis 1:
summed = torch.sum(masked_embeddings, 1)
summed.shape

torch.Size([1, 768])

In [34]:
mask.shape # starting len(text) is non-zero rest are zero

torch.Size([1, 128, 768])

So how we are going to calculate the Mean
we first added by collapsing all columns into 1 row which means we added the first column of all 128 tokens basically we are merging all the token together
Now to take the mean we have to divide this sum by: 
after merging 128x768 into 1x768 matrix we will be dividing each cell by the length of the string because each column before merginf has exactly (11) that much amount of non-zero values

In [36]:
summed_mask = torch.clamp(mask.sum(1), min=1e-9)
summed_mask.shape

torch.Size([1, 768])

In [37]:
mean_pooled = summed / summed_mask

In [39]:
mean_pooled.shape

torch.Size([1, 768])

## Cosine Similarity

In [40]:
sentences = [
    "Three years later, the coffin was still full of Jello.",
    "The fish dreamed of escaping the fishbowl and into the toilet where he saw his friend go.",
    "The person box was packed with jelly many dozens of months later.",
    "Standing on one's head at job interviews forms a lasting impression.",
    "It took him a month to finish the meal.",
    "He found a leprechaun in his walnut shell."
]

# thanks to https://randomwordgenerator.com/sentence.php

In [41]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/bert-base-nli-mean-tokens')
model = AutoModel.from_pretrained('sentence-transformers/bert-base-nli-mean-tokens')

# initialize dictionary that will contain tokenized sentences
tokens = {'input_ids': [], 'attention_mask': []}

for sentence in sentences:
    # tokenize sentence and append to dictionary lists
    new_tokens = tokenizer.encode_plus(sentence, max_length=128, truncation=True,
                                       padding='max_length', return_tensors='pt')
    tokens['input_ids'].append(new_tokens['input_ids'][0])
    tokens['attention_mask'].append(new_tokens['attention_mask'][0])

# reformat list of tensors into single tensor
tokens['input_ids'] = torch.stack(tokens['input_ids'])
tokens['attention_mask'] = torch.stack(tokens['attention_mask'])

In [42]:
tokens['input_ids'].shape

torch.Size([6, 128])

In [43]:
outputs = model(**tokens)
outputs.keys()

odict_keys(['last_hidden_state', 'pooler_output'])

In [44]:
embeddings = outputs.last_hidden_state
embeddings

tensor([[[-6.9229e-02,  6.2300e-01,  3.5370e-02,  ...,  8.0334e-01,
           1.6314e+00,  3.2812e-01],
         [ 3.6729e-02,  6.8419e-01,  1.9460e-01,  ...,  8.4759e-02,
           1.4747e+00, -3.0080e-01],
         [-1.2141e-02,  6.5431e-01, -7.2717e-02,  ..., -3.2600e-02,
           1.7717e+00, -6.8121e-01],
         ...,
         [ 1.9532e-01,  1.1085e+00,  3.3905e-01,  ...,  1.2826e+00,
           1.0114e+00, -7.2755e-02],
         [ 9.0217e-02,  1.0288e+00,  3.2973e-01,  ...,  1.2940e+00,
           9.8651e-01, -1.1125e-01],
         [ 1.2404e-01,  9.7365e-01,  3.9329e-01,  ...,  1.1359e+00,
           8.7685e-01, -1.0435e-01]],

        [[-3.2124e-01,  8.2512e-01,  1.0554e+00,  ..., -1.8555e-01,
           1.5169e-01,  3.9366e-01],
         [-7.1457e-01,  1.0297e+00,  1.1217e+00,  ...,  3.3118e-02,
           2.3820e-01, -1.5632e-01],
         [-2.3522e-01,  1.1353e+00,  8.5941e-01,  ..., -4.3096e-01,
          -2.7242e-02, -2.9676e-01],
         ...,
         [-5.4000e-01,  3

In [45]:
embeddings.shape

torch.Size([6, 128, 768])

In [46]:
attention_mask = tokens['attention_mask']
attention_mask.shape

torch.Size([6, 128])

In [47]:
mask = attention_mask.unsqueeze(-1).expand(embeddings.size()).float()
mask.shape

torch.Size([6, 128, 768])

In [48]:
mask

tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 

In [49]:
masked_embeddings = embeddings * mask
masked_embeddings.shape

torch.Size([6, 128, 768])

In [50]:
masked_embeddings

tensor([[[-0.0692,  0.6230,  0.0354,  ...,  0.8033,  1.6314,  0.3281],
         [ 0.0367,  0.6842,  0.1946,  ...,  0.0848,  1.4747, -0.3008],
         [-0.0121,  0.6543, -0.0727,  ..., -0.0326,  1.7717, -0.6812],
         ...,
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000, -0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000, -0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000, -0.0000]],

        [[-0.3212,  0.8251,  1.0554,  ..., -0.1855,  0.1517,  0.3937],
         [-0.7146,  1.0297,  1.1217,  ...,  0.0331,  0.2382, -0.1563],
         [-0.2352,  1.1353,  0.8594,  ..., -0.4310, -0.0272, -0.2968],
         ...,
         [-0.0000,  0.0000,  0.0000,  ...,  0.0000, -0.0000,  0.0000],
         [-0.0000,  0.0000,  0.0000,  ...,  0.0000, -0.0000,  0.0000],
         [-0.0000,  0.0000,  0.0000,  ...,  0.0000, -0.0000,  0.0000]],

        [[-0.7576,  0.8399, -0.3792,  ...,  0.1271,  1.2514,  0.1365],
         [-0.6591,  0.7613, -0.4662,  ...,  0

In [51]:
summed = torch.sum(masked_embeddings, 1)
summed.shape

torch.Size([6, 768])

In [52]:
summed_mask = torch.clamp(mask.sum(1), min=1e-9)
summed_mask.shape

torch.Size([6, 768])

In [53]:
summed_mask

tensor([[15., 15., 15.,  ..., 15., 15., 15.],
        [22., 22., 22.,  ..., 22., 22., 22.],
        [15., 15., 15.,  ..., 15., 15., 15.],
        [16., 16., 16.,  ..., 16., 16., 16.],
        [12., 12., 12.,  ..., 12., 12., 12.],
        [14., 14., 14.,  ..., 14., 14., 14.]])

In [54]:
mean_pooled = summed / summed_mask

In [55]:
mean_pooled

tensor([[ 0.0745,  0.8637,  0.1795,  ...,  0.7734,  1.7247, -0.1803],
        [-0.3715,  0.9729,  1.0840,  ..., -0.2552, -0.2759,  0.0358],
        [-0.5030,  0.7950, -0.1240,  ...,  0.1441,  0.9704, -0.1791],
        [-0.0132,  0.9773,  1.4516,  ..., -0.8462, -1.4004, -0.4118],
        [-0.2019,  0.0597,  0.8603,  ..., -0.0100,  0.8431, -0.0841],
        [-0.2131,  1.0175, -0.8833,  ...,  0.7371,  0.1947, -0.3011]],
       grad_fn=<DivBackward0>)

In [56]:
from sklearn.metrics.pairwise import cosine_similarity

In [57]:
# convert from PyTorch tensor to numpy array
mean_pooled = mean_pooled.detach().numpy()

# calculate
cosine_similarity(
    [mean_pooled[0]],
    mean_pooled[1:]
)

array([[0.3308891 , 0.7219258 , 0.17475498, 0.4470964 , 0.55483633]],
      dtype=float32)

## Similarity with Sentence Transformer

In [58]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('bert-base-nli-mean-tokens')

In [59]:
sentence_embeddings = model.encode(sentences)

In [60]:
sentence_embeddings.shape

(6, 768)

In [61]:
sentence_embeddings

array([[ 0.07446209,  0.86369693,  0.17946309, ...,  0.77344   ,
         1.7247484 , -0.18027468],
       [-0.3714634 ,  0.972901  ,  1.0839928 , ..., -0.25521338,
        -0.27593744,  0.03575889],
       [-0.5029822 ,  0.79498595, -0.12402679, ...,  0.14406319,
         0.9703747 , -0.17911567],
       [-0.01324333,  0.9772864 ,  1.4515934 , ..., -0.8461654 ,
        -1.4004318 , -0.41184372],
       [-0.20192592,  0.05970341,  0.8602743 , ..., -0.01000785,
         0.84306216, -0.08407696],
       [-0.21311834,  1.0174932 , -0.88327706, ...,  0.73710376,
         0.1946913 , -0.30111292]], dtype=float32)

In [62]:
from sklearn.metrics.pairwise import cosine_similarity

In [63]:
cosine_similarity(
    [sentence_embeddings[0]],
    sentence_embeddings[1:]
)

array([[0.33088902, 0.7219258 , 0.17475492, 0.4470963 , 0.5548362 ]],
      dtype=float32)